In [ ]:
## Setup & Data Loading

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Plot styling
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [ ]:
# Load the dataset
FILE_PATH = "seasonal_agriculture_performance_dataset.csv"

df = pd.read_csv(FILE_PATH)
print("Shape of dataset:", df.shape)
df.head()


In [ ]:
## Data Exploration

df.info()

In [ ]:
df.describe().T

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
print("Categorical columns:", cat_cols)

for col in cat_cols:
    print(f"\n--- {col} ({df[col].nunique()} unique values) ---")
    print(df[col].value_counts().head(10))

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing)
print(f"\nTotal missing cells: {df.isnull().sum().sum()} out of {df.shape[0]*df.shape[1]}")

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Farm IDs:", df['Farm_ID'].duplicated().sum())

In [ ]:
## Data Cleaning & Preparation

df_clean = df.copy()

num_cols_with_na = ['Rainfall_mm', 'Soil_Moisture_pct', 'Yield_Tonnes_Ha']

for col in num_cols_with_na:
    df_clean[col] = df_clean.groupby(['Season', 'Crop'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    # fallback: fill any remaining NaNs (rare edge cases) with overall median
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Remaining missing values after imputation:")
print(df_clean.isnull().sum().sum())

In [ ]:
checks = {
    'Soil_pH': (0, 14),
    'Humidity_pct': (0, 100),
    'Disease_Pest_Risk_pct': (0, 100),
}
for col, (lo, hi) in checks.items():
    out_of_range = df_clean[(df_clean[col] < lo) | (df_clean[col] > hi)]
    print(f"{col}: {len(out_of_range)} rows out of expected range [{lo}, {hi}]")

In [ ]:
df_clean['Profit_Margin_pct'] = (df_clean['Profit_INR'] / df_clean['Revenue_INR'].replace(0, np.nan)) * 100
df_clean['Cost_per_Hectare'] = df_clean['Total_Cost_INR'] / df_clean['Farm_Area_Hectares']
df_clean['Revenue_per_Hectare'] = df_clean['Revenue_INR'] / df_clean['Farm_Area_Hectares']
season_order = ['Kharif', 'Rabi', 'Zaid']
df_clean['Season'] = pd.Categorical(df_clean['Season'], categories=season_order, ordered=True)

df_clean.head()

In [ ]:
## Seasonal Performance Comparison

performance_cols = ['Yield_Tonnes_Ha', 'Production_Tonnes', 'Profit_INR', 'Profit_Margin_pct', 'Revenue_per_Hectare']
seasonal_performance = df_clean.groupby('Season')[performance_cols].mean().round(2)
seasonal_performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df_clean, x='Season', y='Yield_Tonnes_Ha', ax=axes[0,0], palette='YlGn')
axes[0,0].set_title('Yield (Tonnes/Ha) by Season')

sns.boxplot(data=df_clean, x='Season', y='Profit_INR', ax=axes[0,1], palette='YlOrBr')
axes[0,1].set_title('Profit (INR) by Season')

sns.barplot(data=df_clean, x='Season', y='Production_Tonnes', ax=axes[1,0],
            estimator=np.mean, palette='Blues', errorbar=None)
axes[1,0].set_title('Average Production (Tonnes) by Season')

sns.barplot(data=df_clean, x='Season', y='Profit_Margin_pct', ax=axes[1,1],
            estimator=np.mean, palette='Purples', errorbar=None)
axes[1,1].set_title('Average Profit Margin (%) by Season')

plt.tight_layout()
plt.show()

In [ ]:
## Seasonal Patterns in Environmental Conditions

env_cols = ['Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Sunlight_Hours_Day', 'Soil_Moisture_pct']
seasonal_env = df_clean.groupby('Season')[env_cols].mean().round(2)
seasonal_env

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(env_cols):
    sns.violinplot(data=df_clean, x='Season', y=col, ax=axes[i], palette='coolwarm')
    axes[i].set_title(f'{col} by Season')

axes[-1].axis('off')  # hide the unused 6th subplot
plt.tight_layout()
plt.show()

In [ ]:
## Resource Usage Across Seasons

resource_cols = ['Water_Used_m3', 'Water_Efficiency_t_per_1000m3', 'Fertilizer_kg_ha', 'Pesticide_Litre_ha']
seasonal_resources = df_clean.groupby('Season')[resource_cols].mean().round(2)
seasonal_resources


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(data=df_clean, x='Season', y='Water_Used_m3', ax=axes[0], estimator=np.mean,
            palette='Blues', errorbar=None)
axes[0].set_title('Avg Water Used (m³) by Season')

sns.barplot(data=df_clean, x='Season', y='Water_Efficiency_t_per_1000m3', ax=axes[1], estimator=np.mean,
            palette='Greens', errorbar=None)
axes[1].set_title('Avg Water Efficiency by Season')

sns.barplot(data=df_clean, x='Season', y='Fertilizer_kg_ha', ax=axes[2], estimator=np.mean,
            palette='Oranges', errorbar=None)
axes[2].set_title('Avg Fertilizer Use (kg/ha) by Season')

plt.tight_layout()
plt.show()


In [ ]:
irrigation_season = pd.crosstab(df_clean['Season'], df_clean['Irrigation_Method'], normalize='index') * 100
irrigation_season.round(1)

In [ ]:
irrigation_season.plot(kind='bar', stacked=True, figsize=(9,6), colormap='tab20')
plt.title('Irrigation Method Share (%) by Season')
plt.ylabel('% of Farms')
plt.xticks(rotation=0)
plt.legend(title='Irrigation Method', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
## Relationships Between Environmental Conditions & Performance

numeric_df = df_clean.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.3)
plt.title('Correlation Heatmap — All Numeric Variables')
plt.tight_layout()
plt.show()


In [ ]:
target_corr = corr[['Yield_Tonnes_Ha', 'Profit_INR']].sort_values('Yield_Tonnes_Ha', ascending=False)
target_corr

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.scatterplot(data=df_clean, x='Rainfall_mm', y='Yield_Tonnes_Ha', hue='Season', alpha=0.5, ax=axes[0])
axes[0].set_title('Rainfall vs Yield')

sns.scatterplot(data=df_clean, x='Fertilizer_kg_ha', y='Yield_Tonnes_Ha', hue='Season', alpha=0.5, ax=axes[1])
axes[1].set_title('Fertilizer Use vs Yield')

sns.scatterplot(data=df_clean, x='Disease_Pest_Risk_pct', y='Profit_INR', hue='Season', alpha=0.5, ax=axes[2])
axes[2].set_title('Disease/Pest Risk vs Profit')

plt.tight_layout()
plt.show()

In [ ]:
## Regional & Crop-wise Seasonal Consistency
# Average yield by State and Season (pivot table)
state_season_yield = df_clean.pivot_table(values='Yield_Tonnes_Ha', index='State', columns='Season', aggfunc='mean').round(2)
state_season_yield

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(state_season_yield, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('Average Yield (Tonnes/Ha) — State x Season')
plt.tight_layout()
plt.show()

In [ ]:
crop_season_profit = df_clean.pivot_table(values='Profit_INR', index='Crop', columns='Season', aggfunc='mean').round(0)
crop_season_profit

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(crop_season_profit, annot=True, fmt='.0f', cmap='RdYlGn')
plt.title('Average Profit (INR) — Crop x Season')
plt.tight_layout()
plt.show()

In [ ]:
## Statistical Significance Testing

from scipy.stats import f_oneway

def anova_by_season(df, col):
    groups = [group[col].dropna().values for _, group in df.groupby('Season', observed=True)]
    f_stat, p_val = f_oneway(*groups)
    return f_stat, p_val

for metric in ['Yield_Tonnes_Ha', 'Profit_INR', 'Water_Efficiency_t_per_1000m3']:
    f_stat, p_val = anova_by_season(df_clean, metric)
    significance = "Statistically significant (p < 0.05)" if p_val < 0.05 else "Not statistically significant (p >= 0.05)"
    print(f"{metric}: F = {f_stat:.3f}, p = {p_val:.5f}  -->  {significance}")

In [ ]:
r, p = stats.pearsonr(df_clean['Rainfall_mm'], df_clean['Yield_Tonnes_Ha'])
print(f"Rainfall vs Yield: r = {r:.3f}, p = {p:.5f}")

r2, p2 = stats.pearsonr(df_clean['Fertilizer_kg_ha'], df_clean['Yield_Tonnes_Ha'])
print(f"Fertilizer vs Yield: r = {r2:.3f}, p = {p2:.5f}")

In [ ]:
## Outlier & Unusual Pattern Detection
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return (series < lower) | (series > upper)

df_clean['Is_Profit_Outlier'] = df_clean.groupby('Season', observed=True)['Profit_INR'].transform(iqr_outliers)

outlier_summary = df_clean.groupby('Season', observed=True)['Is_Profit_Outlier'].sum()
print("Number of profit outliers per season:")
print(outlier_summary)

In [ ]:
df_clean['Is_Loss'] = df_clean['Profit_INR'] < 0
loss_rate = df_clean.groupby('Season', observed=True)['Is_Loss'].mean() * 100
print("Percentage of farms operating at a loss, by season:")
print(loss_rate.round(1))

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x=loss_rate.index, y=loss_rate.values, palette='Reds')
plt.ylabel('% of Farms at a Loss')
plt.title('Share of Loss-Making Farms by Season')
plt.tight_layout()
plt.show()

In [ ]:
## Summary

best_yield_season = seasonal_performance['Yield_Tonnes_Ha'].idxmax()
best_profit_season = seasonal_performance['Profit_INR'].idxmax()
worst_profit_season = seasonal_performance['Profit_INR'].idxmin()
highest_loss_season = loss_rate.idxmax()
most_water_efficient_season = seasonal_resources['Water_Efficiency_t_per_1000m3'].idxmax()

print("KEY INSIGHTS (auto-generated draft — refine with your own observations)")
print("-" * 70)
print(f"1. Highest average yield observed in: {best_yield_season} season")
print(f"2. Highest average profit observed in: {best_profit_season} season")
print(f"3. Lowest average profit observed in: {worst_profit_season} season")
print(f"4. Highest share of loss-making farms in: {highest_loss_season} season "
      f"({loss_rate[highest_loss_season]:.1f}% of farms)")
print(f"5. Best water-use efficiency observed in: {most_water_efficient_season} season")
print(f"6. Rainfall-Yield correlation: r = {r:.2f} "
      f"({'positive' if r > 0 else 'negative'} relationship, "
      f"{'significant' if p < 0.05 else 'not significant'})")
print(f"7. Fertilizer-Yield correlation: r = {r2:.2f} "
      f"({'positive' if r2 > 0 else 'negative'} relationship, "
      f"{'significant' if p2 < 0.05 else 'not significant'})")